# Домашнее задание 2 (hw_2)

Потребуются следующие данные:

- [коллекция WikIR/en1k](https://zenodo.org/record/3565761/files/wikIR1k.zip)

- Л.Н. Толстой, "Война и мир":
  - [том 1](http://az.lib.ru/t/tolstoj_lew_nikolaewich/text_0040.shtml)
  - [том 2](http://az.lib.ru/t/tolstoj_lew_nikolaewich/text_0050.shtml)
  - [том 3](http://az.lib.ru/t/tolstoj_lew_nikolaewich/text_0060.shtml)
  - [том 4](http://az.lib.ru/t/tolstoj_lew_nikolaewich/text_0070.shtml)

- Ф.М. Достоевский, "Братья Карамазовы":
  - [часть 1](http://az.lib.ru/d/dostoewskij_f_m/text_0100.shtml)
  - [часть 2](http://az.lib.ru/d/dostoewskij_f_m/text_0110.shtml)
  - [часть 3](http://az.lib.ru/d/dostoewskij_f_m/text_0120.shtml)
  - [часть 4](http://az.lib.ru/d/dostoewskij_f_m/text_0130.shtml)

Коллекция WikIR загружается отдельно (более 100 МБ), остальные файлы находятся в
директории `data/`.

Для работы с лемматизацией необходимо установить модель `spaCy`:

```bash
!python -m spacy download en_core_web_sm
```


## Импорт библиотек и загрузка данных

In [1]:
import time
from collections import defaultdict
import os

import ir_measures
import numpy as np
import pandas as pd
import spacy
from ir_measures import AP, MAP, P, nDCG
from nltk.stem.porter import PorterStemmer
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer

documents = pd.read_csv('../data/en1k/documents.csv')


def load_split(split_name):
    queries = pd.read_csv(f'../data/{split_name}/queries.csv')

    qrels = pd.read_csv(
        f'../data/{split_name}/qrels',
        sep='\t',
        header=None,
        names=['query_id', 'Q0', 'doc_id', 'relevance'],
    )
    qrels = qrels[['query_id', 'doc_id', 'relevance']]

    return queries, qrels


queries_train, qrels_train = load_split('training')
queries_val, qrels_val = load_split('validation')
queries_test, qrels_test = load_split('test')

for df in [documents, queries_train, queries_val, queries_test]:
    df.rename(columns={'id_left': 'query_id', 'text_left': 'text'}, inplace=True)
    df.rename(columns={'id_right': 'doc_id', 'text_right': 'text'}, inplace=True)

for df in [qrels_train, qrels_val, qrels_test]:
    df.rename(columns={'id_left': 'query_id'}, inplace=True)


def build_qrels_dict(qrels_df):
    d = defaultdict(list)
    for _, row in qrels_df.iterrows():
        d[row['query_id']].append((row['doc_id'], row['relevance']))
    return d


qrels_dict_test = build_qrels_dict(qrels_test)
queries = queries_test
qrels_dict = qrels_dict_test

documents.tail(), queries.tail()

(         doc_id                                               text
 369716    59396  the population was 416 at the 2010 census the ...
 369717  1950034  the surface of the river is frozen from novemb...
 369718  1984468  the first anti thrombin aptamer tba was genera...
 369719    33966  state of oklahoma as of the 2010 census the po...
 369720  1230943  geetha jeevan born 6 may 1970 is the current m...,
     query_id          text
 95    679227      hiv aids
 96   2136797  maren morris
 97      5622         homer
 98   1313598    south pole
 99    712704   west indies)

## 1. Базовые статистики по запросам

In [2]:
num_queries = len(queries)

query_lengths = queries['text'].apply(lambda x: len(x.split()))
avg_query_len = query_lengths.mean()

relevant_per_query = [len(qrels_dict[qid]) for qid in queries['query_id']]

avg_rel_docs = np.mean(relevant_per_query)

print(f'Количество запросов: {num_queries}')
print(f'Средняя длина запроса: {avg_query_len:.2f}')
print(f'Среднее число релевантных документов: {avg_rel_docs:.2f}')

Количество запросов: 100
Средняя длина запроса: 2.07
Среднее число релевантных документов: 44.35


In [3]:
# 50_000 документов для ускорения экспериментов
documents = documents.sample(n=50_000)

# предобработка текста
stemmer = PorterStemmer()
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])


def tokenize(text):
    return text.lower().split()


def stem_text(tokens):
    return [stemmer.stem(t) for t in tokens]


def lemmatize_text(text):
    doc = nlp(text)
    return [t.lemma_.lower() for t in doc if not t.is_space]


# подготовка коллекций
docs_tokens = [tokenize(doc) for doc in documents['text']]
docs_stemmed = [stem_text(doc) for doc in docs_tokens]

docs_lemmatized = []

for doc in nlp.pipe(documents['text'], batch_size=1000):
    docs_lemmatized.append([t.lemma_.lower() for t in doc if not t.is_space])

## 2. Поиск: TF-IDF и BM25

### Замер времени выполнения

In [4]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(documents['text'])


def search_tfidf(query, top_k=20):
    q_vec = tfidf_vectorizer.transform([query])
    scores = (tfidf_matrix @ q_vec.T).toarray().ravel()
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(documents.iloc[i]['doc_id'], scores[i]) for i in top_idx]


bm25 = BM25Okapi(docs_tokens)
bm25_stem = BM25Okapi(docs_stemmed)
bm25_lemma = BM25Okapi(docs_lemmatized)


def search_bm25(model, query_tokens, top_k=20):
    scores = model.get_scores(query_tokens)
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(documents.iloc[i]['doc_id'], scores[i]) for i in top_idx]


def run_search(queries_df, model_type='bm25', variant='orig'):
    results = []

    start = time.time()

    for _, row in queries_df.iterrows():
        qid = row['query_id']
        qtext = row['text']

        if model_type == 'tfidf':
            res = search_tfidf(qtext)

        else:
            tokens = tokenize(qtext)

            if variant == 'stem':
                tokens = stem_text(tokens)
                model = bm25_stem
            elif variant == 'lemma':
                tokens = lemmatize_text(qtext)
                model = bm25_lemma
            else:
                model = bm25

            res = search_bm25(model, tokens)

        for rank, (doc_id, score) in enumerate(res, 1):
            results.append((qid, doc_id, rank, score))

    end = time.time()

    return results, end - start


runs = {}

for model in ['tfidf', 'bm25']:
    for variant in ['orig', 'stem', 'lemma']:
        if model == 'tfidf' and variant != 'orig':
            continue

        key = f'{model}_{variant}'
        res, t = run_search(queries_test, model, variant)
        runs[key] = (res, t)

        print(f'{key}: {t:.2f} sec')

tfidf_orig: 1.86 sec
bm25_orig: 4.49 sec
bm25_stem: 4.36 sec
bm25_lemma: 4.36 sec


### Сохранение в формате TREC

In [5]:
os.makedirs('results', exist_ok=True)


def save_trec(run, filename):
    with open(filename, 'w') as f:
        for qid, doc_id, rank, score in run:
            f.write(f'{qid} Q0 {doc_id} {rank} {score} run\n')


# теперь можно безопасно сохранять все ранги
for name, (run, _) in runs.items():
    save_trec(run, f'results/{name}.txt')

## 3. Оценка качества (ir-measures)

In [6]:
# преобразуем qrels
qrels_df = pd.read_csv(
    '../data/en1k/test/qrels',
    sep='\t',
    header=None,
    names=['query_id', 'zero', 'doc_id', 'relevance'],
)[['query_id', 'doc_id', 'relevance']]

qrels_ir = [
    ir_measures.Qrel(str(qid), str(doc_id), rel)
    for qid, doc_id, rel in zip(
        qrels_df['query_id'], qrels_df['doc_id'], qrels_df['relevance']
    )
]


def evaluate_run(run):
    run_ir = [
        ir_measures.ScoredDoc(str(qid), str(doc_id), score)
        for qid, doc_id, _, score in run
    ]

    metrics = ir_measures.calc_aggregate(
        [P @ 1, P @ 10, P @ 20, MAP, nDCG @ 20], qrels_ir, run_ir
    )
    return metrics


results_table = []

for name, (run, _) in runs.items():
    metrics = evaluate_run(run)

    row = {'model': name}
    row.update(metrics)

    results_table.append(row)

df_results = pd.DataFrame(results_table)
df_results

,model,P@20,AP,P@10,P@1,nDCG@20
0,tfidf_orig,0.0390,0.034395,0.062,0.23,0.095001
1,bm25_orig,0.0455,0.042862,0.069,0.27,0.110678
2,bm25_stem,0.0460,0.041000,0.067,0.27,0.110834
3,bm25_lemma,0.0450,0.041696,0.067,0.27,0.110219


## 4. Анализ результатов

In [7]:
# сложные и простые запросы


def per_query_map(run):
    run_ir = [
        ir_measures.ScoredDoc(str(qid), str(doc_id), float(score))
        for qid, doc_id, _, score in run
    ]
    return ir_measures.calc([ir_measures.AP], qrels_ir, run_ir).per_query


example_run = runs['bm25_orig'][0]
per_query = list(per_query_map(example_run))
per_query_sorted = sorted(per_query, key=lambda x: x.value)

print('Самые сложные запросы:')
for r in per_query_sorted[:5]:
    print(r)

print('\nСамые простые запросы:')
for r in per_query_sorted[-5:]:
    print(r)

Самые сложные запросы:
Metric(query_id='158491', measure=AP, value=0.0)
Metric(query_id='5728', measure=AP, value=0.0)
Metric(query_id='5115', measure=AP, value=0.0)
Metric(query_id='104086', measure=AP, value=0.0)
Metric(query_id='145194', measure=AP, value=0.0)

Самые простые запросы:
Metric(query_id='2136797', measure=AP, value=0.16666666666666666)
Metric(query_id='200237', measure=AP, value=0.19999999999999998)
Metric(query_id='13690', measure=AP, value=0.22076023391812863)
Metric(query_id='265104', measure=AP, value=0.30729166666666663)
Metric(query_id='996687', measure=AP, value=0.3888888888888889)


In [8]:
# влияние длины запроса
query_lengths = queries_test['text'].apply(lambda x: len(x.split()))

df_analysis = pd.DataFrame(
    {'query_id': queries_test['query_id'], 'length': query_lengths}
)

per_query_scores = list(per_query_map(runs['bm25_orig'][0]))

scores_dict = {int(r.query_id): r.value for r in per_query_scores}
df_analysis['AP'] = df_analysis['query_id'].map(scores_dict)

print(df_analysis.corr())

          query_id    length        AP
query_id  1.000000 -0.040978  0.092077
length   -0.040978  1.000000  0.127756
AP        0.092077  0.127756  1.000000


In [9]:
def inspect_errors(run, top_k=5):
    qrels_set = {
        (row['query_id'], row['doc_id'])
        for _, row in qrels_test.iterrows()
        if row['relevance'] > 0
    }

    errors = []

    for qid, doc_id, rank, score in run:
        if rank <= top_k and (qid, doc_id) not in qrels_set:
            errors.append((qid, doc_id, rank))

    return errors[:10]


errors = inspect_errors(runs['bm25_orig'][0])

for e in errors:
    print(e)

(158491, np.int64(2261272), 1)
(158491, np.int64(635537), 2)
(158491, np.int64(663828), 3)
(158491, np.int64(1180246), 4)
(158491, np.int64(1093529), 5)
(5728, np.int64(1240125), 1)
(5728, np.int64(130455), 2)
(5728, np.int64(2061848), 3)
(5728, np.int64(1466067), 4)
(5728, np.int64(1285700), 5)


## 5. Подбор параметров BM25 (k1, b)

In [10]:
def evaluate_run_on(qrels_df, run):
    qrels_ir = [
        ir_measures.Qrel(str(qid), str(doc_id), int(rel))
        for qid, doc_id, rel in zip(
            qrels_df['query_id'], qrels_df['doc_id'], qrels_df['relevance']
        )
    ]

    run_ir = [
        ir_measures.ScoredDoc(str(qid), str(doc_id), float(score))
        for qid, doc_id, _, score in run
    ]

    return ir_measures.calc_aggregate([nDCG @ 20], qrels_ir, run_ir)


def grid_search():
    best_score = -1
    best_params = None

    for k1 in [0.5, 1.0, 1.5, 2.0]:
        for b in [0.3, 0.5, 0.75, 1.0]:
            model = BM25Okapi(docs_tokens, k1=k1, b=b)

            run = []
            for _, row in queries_val.iterrows():
                tokens = tokenize(row['text'])
                res = search_bm25(model, tokens)

                for rank, (doc_id, score) in enumerate(res, 1):
                    run.append((row['query_id'], doc_id, rank, score))

            metrics = evaluate_run_on(qrels_val, run)
            score = metrics[nDCG @ 20]

            if score > best_score:
                best_score = score
                best_params = (k1, b)

    return best_params, best_score


best_params, _ = grid_search()

print('Лучшие параметры:', best_params)

# прогоняем на test
bm25_tuned = BM25Okapi(docs_tokens, k1=best_params[0], b=best_params[1])

run_tuned, _ = run_search(queries_test, model_type='bm25', variant='orig')

metrics_tuned = evaluate_run(run_tuned)
print(metrics_tuned)

Лучшие параметры: (0.5, 0.5)
{P@20: 0.04549999999999998, AP: 0.04286155339159377, P@10: 0.06899999999999999, P@1: 0.27, nDCG@20: 0.1106777765054642}


# Доп. задание: анализ текстов Толстого и Достоевского

In [11]:
import re


def split_paragraphs(text):
    paragraphs = re.split(r'\n\s*\n', text)
    return [p.strip() for p in paragraphs if len(p.strip()) > 50]


def load_text(path):
    with open(path, encoding='utf-8') as f:
        return f.read()


tolstoy = load_text('../data/tolstoy.txt')
dostoevsky = load_text('../data/dostoevsky.txt')

tolstoy_par = split_paragraphs(tolstoy)
dostoevsky_par = split_paragraphs(dostoevsky)

print(len(tolstoy_par), len(dostoevsky_par))

428 129


In [12]:
vectorizer = TfidfVectorizer(max_features=5000)

all_paragraphs = tolstoy_par + dostoevsky_par
X = vectorizer.fit_transform(all_paragraphs)

sim = (X @ X.T).toarray()


def top_similar(sim_matrix, idx, top_k=3):
    scores = sim_matrix[idx]
    top = np.argsort(scores)[::-1][1 : top_k + 1]
    return top


print('Пример (Толстой):')
idx = 10
for j in top_similar(sim, idx):
    print(tolstoy_par[j][:200])
    print('---')

Пример (Толстой):
Князь Василий исполнил обещание, данное на вечере у Анны Павловны княгине Друбецкой, просившей его о своем единственном сыне Борисе. О нем было доложено государю, и, не в пример другим, он был перевед
---
Наступило молчание. Графиня глядела на гостью, приятно улыбаясь, впрочем, не скрывая того, что не огорчится теперь нисколько, если гостья поднимется и уедет. Дочь гостьи уже оправляла платье, вопросит
---
У Ростовых, как и всегда по воскресениям, обедал кое-кто из близких знакомых.
   Пьер приехал раньше, чтобы застать их одних.
   Пьер за этот год так потолстел, что он был бы уродлив, ежели бы он не б
---
